# Autoencoders and the VAE

Every network so far predicted a label. This one rebuilds its own input through a narrow bottleneck, learning a compact code for the data. A plain autoencoder reconstructs well but maps poorly for new images. A variational autoencoder (VAE) makes the map smooth enough to sample from. Both train on 5,000 MNIST digits with a 2-number latent space.

Order: intuition, maths, cost, from scratch, library, tuning, breaks.

In [ ]:
from common.dl import get_device, set_seed, setup_keras

setup_keras("torch")
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn

from common.datasets import load_mnist

SEED = 42
set_seed(SEED)
np.set_printoptions(precision=3, suppress=True)
plt.rcParams["figure.figsize"] = (6, 3.5)
device = get_device()
torch.backends.cudnn.deterministic = True  # same seed, same numbers on a GPU
x_tr, y_tr, x_te, y_te = load_mnist(5000, 1000, seed=SEED)
X_tr = torch.from_numpy(x_tr.reshape(len(x_tr), -1))
X_te = torch.from_numpy(x_te.reshape(len(x_te), -1))
print("device:", device, "| train", tuple(X_tr.shape))

## 1. Intuition

An **autoencoder** encoder squeezes 784 pixels to 2 numbers; the decoder rebuilds them. The two numbers must hold what matters, or the rebuild drifts.

The 2D code is not tidy. The encoder leaves holes that decode to rubbish. A **VAE** encodes each image as a cloud, not a point, and pulls the clouds towards one standard normal. Points near the origin then decode to digit-like outputs, and new digits can be sampled from $\mathcal N(0,I)$.

## 2. Maths and cost function

The encoder outputs a mean $\boldsymbol\mu$ and a log-variance $\log\boldsymbol\sigma^2$, then samples

$$\mathbf z=\boldsymbol\mu+\boldsymbol\sigma\odot\boldsymbol\epsilon,\qquad \boldsymbol\epsilon\sim\mathcal N(0,I)$$

This **reparameterisation trick** moves the randomness into $\boldsymbol\epsilon$, so backprop flows through $\boldsymbol\mu$ and $\boldsymbol\sigma$. The loss has two parts:

$$L=\underbrace{-\log p(\mathbf x\mid\mathbf z)}_{\text{reconstruction}}+\beta\,\underbrace{\tfrac12\sum_j\big(\mu_j^2+\sigma_j^2-\log\sigma_j^2-1\big)}_{\mathrm{KL}\big(q(\mathbf z\mid\mathbf x)\,\|\,\mathcal N(0,I)\big)}$$

Reconstruction is binary cross-entropy over the 784 pixels. The KL term prices straying from the standard normal; at $\beta=1$ the loss is the evidence lower bound.

## 3. From scratch

One class covers both: `variational=False` drops the KL term. The loop follows notebook 08.

In [ ]:
class Autoencoder(nn.Module):
    """Squeeze 784 pixels through ``latent`` numbers and rebuild them."""

    def __init__(self, latent: int = 2, variational: bool = False) -> None:
        super().__init__()
        self.variational = variational
        self.enc = nn.Sequential(nn.Linear(784, 256), nn.ReLU())
        self.mu = nn.Linear(256, latent)
        self.logvar = nn.Linear(256, latent) if variational else None
        self.dec = nn.Sequential(nn.Linear(latent, 256), nn.ReLU(), nn.Linear(256, 784))

    def encode(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor | None]:
        h = self.enc(x)
        return self.mu(h), (self.logvar(h) if self.variational else None)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor | None]:
        mu, logvar = self.encode(x)
        z = mu
        if logvar is not None:  # the reparameterisation trick
            z = mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)
        return self.dec(z), mu, logvar


def loss_terms(x: torch.Tensor, recon: torch.Tensor, mu: torch.Tensor, logvar: torch.Tensor | None):
    """Reconstruction (summed over pixels) and KL to N(0, I), both averaged over the batch."""
    rec = nn.functional.binary_cross_entropy_with_logits(recon, x, reduction="none").sum(1).mean()
    kl = (
        torch.zeros(())
        if logvar is None
        else (-0.5 * (1 + logvar - mu**2 - logvar.exp()).sum(1)).mean()
    )
    return rec, kl.to(rec.device)


def fit(
    model: Autoencoder, epochs: int = 12, beta: float = 1.0, seed: int = SEED
) -> list[tuple[float, float]]:
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    g = torch.Generator().manual_seed(seed)
    hist = []
    for _ in range(epochs):
        order = torch.randperm(len(X_tr), generator=g)
        tot_r = tot_k = 0.0
        for i in range(0, len(order), 100):
            xb = X_tr[order[i : i + 100]].to(device)
            rec, kl = loss_terms(xb, *model(xb))
            opt.zero_grad()
            (rec + beta * kl).backward()
            opt.step()
            tot_r, tot_k = tot_r + rec.item() * len(xb), tot_k + kl.item() * len(xb)
        hist.append((tot_r / len(order), tot_k / len(order)))
    return hist


def evaluate(model: Autoencoder) -> tuple[float, float]:
    model.eval()
    with torch.no_grad():
        return tuple(v.item() for v in loss_terms(X_te.to(device), *model(X_te.to(device))))


set_seed(SEED)
ae = Autoencoder(2)
h = fit(ae)
print(
    f"autoencoder: train reconstruction loss {h[0][0]:.0f} -> {h[-1][0]:.0f}, test {evaluate(ae)[0]:.0f}"
)

The autoencoder cuts training reconstruction loss from 267 to 166 nats (summed over 784 pixels) in 12 epochs, and scores 164 on the test set.

## 4. Library version

The same loop with `variational=True`.

In [ ]:
set_seed(SEED)
vae = Autoencoder(2, variational=True)
h = fit(vae)
rec, kl = evaluate(vae)
print(f"VAE: test reconstruction {rec:.0f}, KL {kl:.1f}, total {rec + kl:.0f}")


def latent_plot(model: Autoencoder, ax: plt.Axes, title: str) -> None:
    model.eval()
    with torch.no_grad():
        z = model.encode(X_te.to(device))[0].cpu().numpy()
    ax.scatter(z[:, 0], z[:, 1], c=y_te, cmap="tab10", s=6)
    ax.set_title(title)


for name, m in (("autoencoder", ae), ("VAE", vae)):
    z = m.encode(X_te.to(device))[0].detach().cpu().numpy()
    print(
        f"{name}: latent std per dimension {z.std(0)}, 99th percentile of |z| {np.percentile(np.abs(z), 99):.1f}"
    )
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
latent_plot(ae, ax[0], "autoencoder latent space")
latent_plot(vae, ax[1], "VAE latent space")
plt.show()

The VAE reconstructs about as well (166) and pays 5.2 nats of KL, a total test loss of 172. The autoencoder's codes spread over $\pm10$ (99% below 10 in absolute value), irregular in outline. The VAE's sit in a tighter blob (99% below 4) near the origin, standard deviation 1.7 and 1.1 on the two axes, closer to the $\mathcal N(0,I)$ target but not equal. Labels were never used in training.

Now decode a 10x10 grid of latent points. The autoencoder grid spans the range its codes use. The VAE grid spans $[-2.5, 2.5]$.

In [ ]:
def decode_grid(model: Autoencoder, lim: float, n: int = 10) -> np.ndarray:
    zs = np.linspace(-lim, lim, n)
    z = torch.tensor([[a, b] for b in zs[::-1] for a in zs], dtype=torch.float32, device=device)
    with torch.no_grad():
        imgs = torch.sigmoid(model.dec(z)).cpu().numpy().reshape(n, n, 28, 28)
    return imgs.transpose(0, 2, 1, 3).reshape(n * 28, n * 28)


zs_ae = ae.encode(X_te.to(device))[0].detach().cpu().numpy()
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(decode_grid(ae, np.abs(zs_ae).max() * 0.8), cmap="gray")
ax[0].set_title("autoencoder: decode a grid of latent points")
ax[1].imshow(decode_grid(vae, 2.5), cmap="gray")
ax[1].set_title("VAE: decode a grid over [-2.5, 2.5]")
for a in ax:
    a.axis("off")
plt.show()
set_seed(SEED)
z = torch.randn(5, 2, device=device)
print(
    "five samples drawn from N(0, I) decoded by the VAE: pixel mean",
    torch.sigmoid(vae.dec(z)).mean().item(),
)

The autoencoder's grid has broken shapes in the top left, where few codes landed, and sharp digits elsewhere. The VAE grid morphs smoothly from a 0 through 7s and 1s to 6s and 9s, no junk cells, though blurrier. That is the VAE bargain.

## 5. Tuning

$\beta$ weights the KL term. Five values, 8 epochs.

In [ ]:
for beta in (0.0, 0.1, 1.0, 10.0, 100.0):
    set_seed(SEED)
    m = Autoencoder(2, variational=True)
    fit(m, epochs=8, beta=beta)
    rec, kl = evaluate(m)
    print(f"beta {beta:>5}: test reconstruction {rec:6.1f}, KL {kl:6.2f}")

At $\beta=0$ there is no pull and the KL is 36.7 nats. As $\beta$ rises the KL falls (16.6, 5.7, 1.5, 0.0) and reconstruction loss rises from 168 to 206. A code closer to the prior is easier to sample and worse at reconstruction.

## 6. What breaks

At $\beta=100$ the KL is 0.00. Every input maps to the same cloud, the standard normal, and the image is ignored. This is **posterior collapse**.

In [ ]:
for beta in (1.0, 100.0):
    set_seed(SEED)
    m = Autoencoder(2, variational=True)
    fit(m, epochs=8, beta=beta)
    with torch.no_grad():
        mu = m.encode(X_te.to(device))[0]
        out = torch.sigmoid(m.dec(mu)).cpu().numpy()
    print(
        f"beta {beta}: latent std per dimension {mu.std(0).cpu().numpy()}, spread of reconstructions {out.std(0).mean():.3f}"
    )

At $\beta=100$ the latent means have a standard deviation of 0.004 and 0.005, against 1.8 and 1.1 at $\beta=1$. Every test digit reconstructs to the same image (output spread 0.000, against 0.081). Collapse can also strike at $\beta=1$ with a powerful decoder. A KL near zero with good reconstructions means the latent is ignored.